# 04: Conceptual Framework Theory

**Purpose:** Test the Conceptual Framework Theory of Being of Light encounters — that an objective phenomenon is named with culturally available vocabulary ("constant state, variable form").

**Research Questions:**
1. Do Being of Light encounters reflect Western monotheistic conceptual frameworks?
2. Does the experience CORRECT prior beliefs or merely CONFIRM them?
3. What is the relationship between cultural concepts and actual phenomenology?
4. Can we distinguish between projection and genuine encounter?

**Dataset:** 6,751 NDEs (IANDS + NDERF; 2 exact duplicates removed) · **Schema:** 2026-01-06 · **Audited and corrected:** 2026-10-05

Religion in this notebook is `religious_belief_at_nde` (belief at the time of the NDE); notebook 01 uses `religious_background`. Both are reported where they differ materially.

## Audit corrections (2026-10-05)

The previous version of this notebook could not support its conclusions: many cells read **legacy (v1) field names that no longer exist** and therefore evaluated to 0%, while the narrative text printed "✓ VALIDATED" regardless. The final summary cells printed **hard-coded percentages** (18.7%, 41.6%, 89.9%, 38.7%, 57.8%, 41.8%, 5.5%, 23.1%, 14.4%, 27.2%) that were not computed anywhere in the notebook.

| Section | Problem | Effect | Fix |
|---|---|---|---|
| Part I | `guidance_level`, `communication_mode` (singular), greeting `'love_warmth'` do not exist | Guidance 0.0%, greeting 0.0%, yet "UNIQUE AUTHORITATIVE POSITION" printed | Schema fields |
| Part II | `'self_judgment'`, `'guide_or_light'`, `'harsh'` are not `judgment_source` values | Self-judgment 0; "LOVE DOMINANT over shame" printed while love (163) < shame (169) | Schema values; honest comparison |
| Part III | Communication/guidance fields wrong | "0.0% active communication" followed by "PERSONAL, CONSCIOUS ENTITY" | Schema fields; personhood stated as interpretation |
| Part IV | `str.contains('atheist')` and `('agnostic')` both matched `atheist_agnostic` | Same 429 records counted twice | Exact matching |
| Part IV | "Spirituality ↑ while Religiosity ↓" printed without a test | Religiosity shows no net change | Paired tests |
| Part V (hell) | Harsh judgment searched in `judgment_source` | "Hell virtually ABSENT … proves" | Hellish realms, distressing valence, harsh intensity counted |
| Part V (Jesus vs Unknown) | 3 of 6 compared properties were broken (0% vs 0%); the rest used all encounters as denominator (e.g. "loving judgment 4.9% vs 5.0%") | "All differences < 10%" was an artifact | Proper denominators, Fisher tests, Holm correction |
| Part VI (ML) | Target God/Jesus vs *everything else*, including 1,898 other-being-only cases; feature `single_being` mechanically related to the target; then "is_christian is TOP predictor" although `single_being` was (0.85) | Leakage; contradictory text | Replaced by a clean model |
| Normative path | `return_choice`, `return_reason`, `belief_change`, `spirituality_shift` are v1 fields; summary numbers hard-coded | All 0% / fabricated | See notebook 02 for the corrected analysis |

## Setup

In [1]:
import sys, json, warnings
from pathlib import Path
from collections import Counter

import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from scipy.stats import chi2_contingency, fisher_exact, mannwhitneyu, wilcoxon
from statsmodels.stats.multitest import multipletests

NDE_ROOT = Path.cwd().parent
sys.path.insert(0, str(NDE_ROOT / "scripts"))
import nde_dataset as nd  # verified loader: schema-checked fields, dedup, narrative word counts

OUTPUT_DIR = NDE_ROOT / "output"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
sns.set_style("whitegrid")
plt.rcParams["figure.figsize"] = (14, 8)
pd.set_option("display.width", 160)
pd.set_option("display.max_columns", 30)

df = nd.load_frame()          # one row per NDE; exact duplicate narratives removed
N = len(df)
print(f"Loaded {N:,} records (after removing exact duplicate narratives)")
print(df["dataset"].value_counts().to_string())


lb = df[df["has_light_being"]].copy()
other_only = df[df["has_other_being"] & ~df["has_light_being"]].copy()
NAMED = set(nd.RELIGIOUS_FIGURE_IDS)
lb["unknown_only"] = lb["being_identifications"].apply(lambda ids: "unknown_presence" in ids and not set(ids) & NAMED)
def hierarchical(ids):
    if "god" in ids: return "god"
    if "jesus" in ids: return "jesus"
    if set(ids) & {"religious_figure_specified", "buddha"}: return "religious_figure"
    return "unknown_presence"
lb["light_being_primary"] = lb["being_identifications"].apply(hierarchical)

Loaded 6,751 records (after removing exact duplicate narratives)
dataset
nderf    5659
iands    1092


---
## Part I: The Being of Light across religious belief at the time of the NDE

In [2]:
print(f"Light-Being encounters: {nd.fmt_pct(len(lb), N)};  other beings only: {nd.fmt_pct(len(other_only), N)}")
print("\nPrimary identification (hierarchical: god > jesus > named figure > unknown):")
for k, v in lb["light_being_primary"].value_counts().items():
    print(f"  {k:18s} {nd.fmt_pct(int(v), len(lb))}")

rows = []
for rel, g in lb.groupby("religious_belief_at_nde"):
    if rel == "not_mentioned" or len(g) < 10:
        continue
    rows.append({"belief at NDE": rel, "n": len(g), "unknown only %": round(100*g.unknown_only.mean(), 1),
                 "god %": round(100*(g.light_being_primary=="god").mean(), 1), "jesus %": round(100*(g.light_being_primary=="jesus").mean(), 1),
                 "named figure %": round(100*(g.light_being_primary=="religious_figure").mean(), 1)})
print("\nIdentification by belief at the time of the NDE (n ≥ 10):")
print(pd.DataFrame(rows).sort_values("n", ascending=False).to_string(index=False))

test = lb[lb["religious_belief_at_nde"].isin(["christian", "atheist_agnostic", "other", "spiritual_not_religious"])]
t = pd.crosstab(test["religious_belief_at_nde"], test["unknown_only"])
chi2, p, dof, _ = chi2_contingency(t)
print(f"\nBelief × 'unknown presence only': χ² = {chi2:.2f}, df = {dof}, p = {p:.2g}, Cramér's V = {nd.cramers_v(t):.3f}  ({nd.expected_count_report(t)})")

Light-Being encounters: 1881/6751 = 27.9% [95% CI 26.8–28.9];  other beings only: 1898/6751 = 28.1% [95% CI 27.1–29.2]

Primary identification (hierarchical: god > jesus > named figure > unknown):
  unknown_presence   951/1881 = 50.6% [95% CI 48.3–52.8]
  god                499/1881 = 26.5% [95% CI 24.6–28.6]
  jesus              309/1881 = 16.4% [95% CI 14.8–18.2]
  religious_figure   122/1881 = 6.5% [95% CI 5.5–7.7]

Identification by belief at the time of the NDE (n ≥ 10):
          belief at NDE   n  unknown only %  god %  jesus %  named figure %
              christian 460            38.0   30.0     24.6             7.4
       atheist_agnostic 124            62.1   18.5     13.7             5.6
                  other  84            64.3   14.3     10.7            10.7
spiritual_not_religious  26            61.5   30.8      3.8             3.8
               buddhist  10            40.0   50.0      0.0            10.0

Belief × 'unknown presence only': χ² = 38.40, df = 3, p = 2.3e

**Finding (statistically supported):** belief at the time of the NDE is associated with naming. Christians identify an "unknown presence only" in ~38% of Light-Being encounters versus ~60% for atheist/agnostic, SBNR and "other" experiencers; the effect is small-to-moderate (Cramér's V ≈ 0.2). Every group uses "unknown presence" for a large share of encounters.

*Not testable here:* whether the Being is experienced as *one* entity. The schema records types of beings, not their number (see notebook 01, where 2 of the 8 Hindu/Buddhist cases name several divine figures).

---
## Part II: Love vs judgment across ALL life reviews

In [3]:
rev = df[df["life_review"]].copy()
print(f"Life reviews: {nd.fmt_pct(len(rev), N)}")
cls = rev[rev["judgment_source"] != "not_mentioned"]
print("\nJudgment source (classifiable):")
for k, v in cls["judgment_source"].value_counts().items():
    print(f"  {k:18s} {nd.fmt_pct(int(v), len(cls))}")
print(f"  → No external evaluator (none + self): {nd.fmt_pct(int(cls.judgment_source.isin(['none','self']).sum()), len(cls))}")

spec = rev[rev["life_review_emotional_tone"] != "not_specified"]
print("\nExperiencer emotional tone (specified):")
for k, v in spec["life_review_emotional_tone"].value_counts().items():
    print(f"  {k:16s} {nd.fmt_pct(int(v), len(spec))}")
love, shame = int((rev.life_review_emotional_tone == "love").sum()), int((rev.life_review_emotional_tone == "shame_or_regret").sum())
print(f"  love : shame_or_regret = {love} : {shame} = {love/shame:.2f}  → love is NOT dominant over shame")
print(f"\nEmpathetic perspective of others: {nd.fmt_pct(int(nd.isin(rev, 'perspective_of_others', nd.YES).sum()), len(rev))}")

print("\nBy belief at the time of the NDE (n ≥ 10 life reviews):")
rows = []
for rel, g in rev.groupby("religious_belief_at_nde"):
    if rel == "not_mentioned" or len(g) < 10:
        continue
    c = g[g.judgment_source != "not_mentioned"]
    r = g[g.judgment_intensity.isin(["loving_gentle", "neutral", "uncomfortable", "harsh_condemning"])]
    rows.append({"belief": rel, "n": len(g), "no external (classifiable) %": round(100*c.judgment_source.isin(["none", "self"]).mean(), 1),
                 "loving (rated) %": round(100*(r.judgment_intensity == "loving_gentle").mean(), 1) if len(r) else np.nan,
                 "rated n": len(r), "harsh n": int((r.judgment_intensity == "harsh_condemning").sum()),
                 "love tone n": int((g.life_review_emotional_tone == "love").sum()),
                 "shame tone n": int((g.life_review_emotional_tone == "shame_or_regret").sum())})
print(pd.DataFrame(rows).sort_values("n", ascending=False).to_string(index=False))
ch = rev[(rev.religious_belief_at_nde == "christian") & rev.judgment_intensity.isin(["loving_gentle", "neutral", "uncomfortable", "harsh_condemning"])]
at = rev[(rev.religious_belief_at_nde == "atheist_agnostic") & rev.judgment_intensity.isin(["loving_gentle", "neutral", "uncomfortable", "harsh_condemning"])]
t = [[int((ch.judgment_intensity == "loving_gentle").sum()), int((ch.judgment_intensity != "loving_gentle").sum())],
     [int((at.judgment_intensity == "loving_gentle").sum()), int((at.judgment_intensity != "loving_gentle").sum())]]
print(f"\nLoving share of rated evaluations, Christian vs atheist/agnostic: {t[0][0]}/{len(ch)} vs {t[1][0]}/{len(at)}; Fisher p = {fisher_exact(t)[1]:.3f}")

Life reviews: 1183/6751 = 17.5% [95% CI 16.6–18.4]

Judgment source (classifiable):
  none               493/928 = 53.1% [95% CI 49.9–56.3]
  guide_or_entity    185/928 = 19.9% [95% CI 17.5–22.6]
  being_of_light     148/928 = 15.9% [95% CI 13.7–18.4]
  self               95/928 = 10.2% [95% CI 8.4–12.4]
  deceased_relative  7/928 = 0.8% [95% CI 0.4–1.5]
  → No external evaluator (none + self): 588/928 = 63.4% [95% CI 60.2–66.4]

Experiencer emotional tone (specified):
  mixed            315/688 = 45.8% [95% CI 42.1–49.5]
  shame_or_regret  169/688 = 24.6% [95% CI 21.5–27.9]
  love             163/688 = 23.7% [95% CI 20.7–27.0]
  neutral          41/688 = 6.0% [95% CI 4.4–8.0]
  love : shame_or_regret = 163 : 169 = 0.96  → love is NOT dominant over shame

Empathetic perspective of others: 213/1183 = 18.0% [95% CI 15.9–20.3]

By belief at the time of the NDE (n ≥ 10 life reviews):
                 belief   n  no external (classifiable) %  loving (rated) %  rated n  harsh n  love tone n 


Loving share of rated evaluations, Christian vs atheist/agnostic: 52/88 vs 20/48; Fisher p = 0.072


**Findings.** Across all life reviews, about two-thirds of classifiable reviews involve no external evaluator (none or self), harsh evaluation is rare, and the experiencer's emotional tone is predominantly *mixed*, with love and shame/regret equally common. The previous printout "LOVE DOMINANT over shame" (when love = 163 < shame = 169) and "Christians … experience overwhelming love/acceptance" are withdrawn. Christians and atheists/agnostics have similar rates of external evaluation and a similar love/shame balance; the loving share of *rated* evaluations is lower among atheists/agnostics (≈42% vs ≈59%), a difference that does not reach significance with 48 rated atheist cases.

---
## Part III: Personal entity vs impersonal force — communication and guidance

In [4]:
def per_case(frame, column, value):
    return 100 * nd.has(frame, column, value).mean()

print(f"Light-Being encounters (n = {len(lb)}): per-experiencer rates")
any_comm = lb["communication_modes"].apply(lambda ms: bool(set(ms) - {"no_communication"}))
print(f"  Any communication recorded:  {nd.fmt_pct(int(any_comm.sum()), len(lb))}")
for m in ["telepathic", "nonverbal", "normal_speech", "no_communication"]:
    print(f"  {m:28s} {per_case(lb, 'communication_modes', m):.1f}%")
print(f"  Guidance received ('yes'):   {nd.fmt_pct(int((lb.guidance_received=='yes').sum()), len(lb))}")
for gtype in ["teaching", "life_guidance", "informational", "directional", "comfort"]:
    print(f"  guidance type {gtype:14s} {per_case(lb, 'guidance_types', gtype):.1f}%")
print(f"  Personified identification (God/Jesus/named figure): {nd.fmt_pct(int((~lb.unknown_only).sum()), len(lb))};  unknown presence only: {nd.fmt_pct(int(lb.unknown_only.sum()), len(lb))}")

Light-Being encounters (n = 1881): per-experiencer rates
  Any communication recorded:  1714/1881 = 91.1% [95% CI 89.8–92.3]
  telepathic                   48.2%
  nonverbal                    40.2%
  normal_speech                30.5%
  no_communication             4.6%
  Guidance received ('yes'):   1537/1881 = 81.7% [95% CI 79.9–83.4]
  guidance type teaching       25.3%
  guidance type life_guidance  36.3%
  guidance type informational  37.8%
  guidance type directional    46.5%
  guidance type comfort        35.7%
  Personified identification (God/Jesus/named figure): 930/1881 = 49.4% [95% CI 47.2–51.7];  unknown presence only: 951/1881 = 50.6% [95% CI 48.3–52.8]


**Finding (descriptive):** almost all Light-Being encounters record some form of communication (most often telepathic or non-verbal), and ~82% record guidance. These are the observable correlates the framework associates with personhood. That the Being *is* a personal, conscious entity is an **interpretation** of these reports, not something the coding can establish; comparisons with other-being encounters, adjusted for narrative length, are in notebook 01, Part III.

---
## Part IV: Belief correction — before/after and expectation

In [5]:
def pre_post(frame, before, after, scale):
    m = frame[before].isin(scale) & frame[after].isin(scale)
    x, y = frame.loc[m, before].map(scale), frame.loc[m, after].map(scale)
    return x, y

print("1. DEATH FEAR (both levels stated)")
x, y = pre_post(df, "death_fear_before", "death_fear_after", nd.FEAR_SCALE)
print(f"  n = {len(x)}; high fear before (significant/severe): {nd.fmt_pct(int((x>=3).sum()), len(x))}; "
      f"none/minimal after: {nd.fmt_pct(int((y<=1).sum()), len(x))}; Wilcoxon p = {wilcoxon(x, y).pvalue:.2g}")

print("\n2. SPIRITUALITY vs RELIGIOSITY (all four levels stated)")
m = (df.spirituality_before.isin(nd.SPIRITUALITY_SCALE) & df.spirituality_after.isin(nd.SPIRITUALITY_SCALE)
     & df.religiosity_before.isin(nd.RELIGIOSITY_SCALE) & df.religiosity_after.isin(nd.RELIGIOSITY_SCALE))
s = df[m]
ds = s.spirituality_after.map(nd.SPIRITUALITY_SCALE) - s.spirituality_before.map(nd.SPIRITUALITY_SCALE)
dr = s.religiosity_after.map(nd.RELIGIOSITY_SCALE) - s.religiosity_before.map(nd.RELIGIOSITY_SCALE)
print(f"  n = {len(s)}; spirituality up {100*(ds>0).mean():.1f}% / down {100*(ds<0).mean():.1f}%; "
      f"religiosity up {100*(dr>0).mean():.1f}% / down {100*(dr<0).mean():.1f}%")
print(f"  spirituality ↑ AND religiosity ↓: {nd.fmt_pct(int(((ds>0)&(dr<0)).sum()), len(s))}")
print(f"  Wilcoxon spirituality p = {wilcoxon(ds).pvalue:.2g}; religiosity p = {wilcoxon(dr[dr!=0]).pvalue if (dr!=0).any() else float('nan'):.2g}")
print(pd.crosstab(np.sign(ds).rename("spirituality change"), np.sign(dr).rename("religiosity change")).to_string())

1. DEATH FEAR (both levels stated)
  n = 327; high fear before (significant/severe): 215/327 = 65.7% [95% CI 60.4–70.7]; none/minimal after: 306/327 = 93.6% [95% CI 90.4–95.8]; Wilcoxon p = 4.9e-46

2. SPIRITUALITY vs RELIGIOSITY (all four levels stated)


  n = 539; spirituality up 82.4% / down 2.0%; religiosity up 30.2% / down 18.6%
  spirituality ↑ AND religiosity ↓: 79/539 = 14.7% [95% CI 11.9–17.9]
  Wilcoxon spirituality p = 5.6e-76; religiosity p = 3.5e-06
religiosity change   -1    0    1
spirituality change              
-1                    7    4    0
 0                   14   69    1
 1                   79  203  162


**Correction:** the "Spirituality ↑ while Religiosity ↓" pattern is not what the data show. Where all four levels are stated, spirituality rises in ~82%; religiosity is unchanged for most and, where it changes, **rises more often than it falls** (≈30% vs ≈19%, p < 0.001). Only ~15% show the inverse (spirituality ↑, religiosity ↓) pattern. Across all records with religiosity before/after stated (n = 1,168; notebook 02) religiosity shows no net change. Either way, a general decline in religiosity is not supported.

In [6]:
print("3. CHANGE BY BELIEF AT THE TIME OF THE NDE (exact category match; previous version double-counted atheist_agnostic)")
rows = []
for rel in ["christian", "atheist_agnostic", "spiritual_not_religious", "jewish", "muslim", "buddhist", "hindu", "other"]:
    g = df[df["religious_belief_at_nde"] == rel]
    fx, fy = pre_post(g, "death_fear_before", "death_fear_after", nd.FEAR_SCALE)
    sx, sy = pre_post(g, "spirituality_before", "spirituality_after", nd.SPIRITUALITY_SCALE)
    rows.append({"belief": rel, "N": len(g), "fear pairs": len(fx), "fear ↓ %": round(100*(fy<fx).mean(), 1) if len(fx) else np.nan,
                 "spirit pairs": len(sx), "spirit ↑ %": round(100*(sy>sx).mean(), 1) if len(sx) else np.nan})
print(pd.DataFrame(rows).to_string(index=False))

print("\n4. 'HELL' AND DISTRESS — are they absent?")
print(f"  Hellish realm:                 {nd.fmt_pct(int(nd.has(df, 'realm_types', 'hellish').sum()), N)}")
print(f"  Distressing/terrifying overall: {nd.fmt_pct(int(df.overall_valence.isin(['distressing','terrifying']).sum()), int((df.overall_valence!='not_mentioned').sum()))} of stated valence")
print(f"  Harsh/condemning judgment:     {int((df.judgment_intensity=='harsh_condemning').sum())} records")
print(f"  Void/darkness realm:           {nd.fmt_pct(int(nd.has(df, 'realm_types', 'void_darkness').sum()), N)}")

3. CHANGE BY BELIEF AT THE TIME OF THE NDE (exact category match; previous version double-counted atheist_agnostic)


                 belief    N  fear pairs  fear ↓ %  spirit pairs  spirit ↑ %
              christian 1127         120      83.3           281        81.1
       atheist_agnostic  429          53      83.0           208        91.3
spiritual_not_religious   81           6      83.3            52        76.9
                 jewish   29           3      33.3             9        66.7
                 muslim   46           4     100.0             6        33.3
               buddhist   26           6      83.3            10        70.0
                  hindu   15           2      50.0             3        33.3
                  other  271          31      93.5            88        79.5

4. 'HELL' AND DISTRESS — are they absent?
  Hellish realm:                 191/6751 = 2.8% [95% CI 2.5–3.3]
  Distressing/terrifying overall: 224/6123 = 3.7% [95% CI 3.2–4.2] of stated valence
  Harsh/condemning judgment:     14 records
  Void/darkness realm:           2084/6751 = 30.9% [95% CI 29.8–32.0]

**Findings.**
- Death-fear reduction and spirituality increase are found in every belief group with enough data.
- **Correction on "hell":** hellish realms are reported in 2.8% of NDEs and distressing/terrifying experiences in 3.7% of accounts with a stated valence. Hell is *uncommon*, not "virtually absent", and its rarity is not "proof" against projection: distressing NDEs are known to be under-reported in self-selected online archives.
- Expectation fields (`personal_expectation_consistency`) are analysed in notebook 01: about half of experiencers stating it say the NDE contradicted or surprised their expectations, equally after Light-Being and other-being encounters.

---
## Part V: Jesus vs Unknown — the critical test

**Hypothesis (framework):** if the Being is the same phenomenon named differently, Christians who name it "Jesus" and Christians who name it "unknown presence" should report the same *properties*.

Groups (Christian belief at the time of the NDE): **Jesus-only** = Jesus identified, neither God nor unknown presence; **Unknown-only** = unknown presence, no named figure. Each property uses the appropriate denominator; p-values are Fisher exact with Holm correction for 11 comparisons.

In [7]:
def property_table(a, b, label_a, label_b):
    defs = {
        "guidance received": (lambda d: d.guidance_received == "yes", None),
        "teaching guidance": (lambda d: nd.has(d, "guidance_types", "teaching"), None),
        "telepathic communication": (lambda d: nd.has(d, "communication_modes", "telepathic"), None),
        "life review": (lambda d: d.life_review, None),
        "loving judgment | rated": (lambda d: d.judgment_intensity == "loving_gentle",
                                    lambda d: d.judgment_intensity.isin(["loving_gentle", "neutral", "uncomfortable", "harsh_condemning"])),
        "love tone | tone specified": (lambda d: d.life_review_emotional_tone == "love",
                                       lambda d: d.life_review & (d.life_review_emotional_tone != "not_specified")),
        "pleasant valence | stated": (lambda d: d.overall_valence.isin(["pleasant", "incredibly_pleasant"]),
                                      lambda d: d.overall_valence != "not_mentioned"),
        "light_encounter = being_of_light": (lambda d: d.light_encounter == "being_of_light", None),
        "unity experience": (lambda d: nd.isin(d, "unity_experience", nd.YES), None),
        "sense of belonging": (lambda d: d.sense_of_belonging.isin(["explicit", "implied"]), None),
        "mission commissioned": (lambda d: nd.isin(d, "mission_commissioned", nd.YES), None),
    }
    rows = []
    for name, (f, mask) in defs.items():
        xa = f(a)[mask(a)] if mask else f(a)
        xb = f(b)[mask(b)] if mask else f(b)
        ka, na_, kb, nb_ = int(xa.sum()), len(xa), int(xb.sum()), len(xb)
        p = fisher_exact([[ka, na_ - ka], [kb, nb_ - kb]])[1]
        d = ka / na_ - kb / nb_
        se = np.sqrt(ka/na_*(1-ka/na_)/na_ + kb/nb_*(1-kb/nb_)/nb_)
        rows.append({"property": name, label_a: f"{ka}/{na_} ({100*ka/na_:.1f}%)", label_b: f"{kb}/{nb_} ({100*kb/nb_:.1f}%)",
                     "diff (pp)": round(100*d, 1), "95% CI (pp)": f"{100*(d-1.96*se):.1f} to {100*(d+1.96*se):.1f}", "p": p})
    t = pd.DataFrame(rows)
    t["p_holm"] = multipletests(t["p"], method="holm")[1]
    return t

christian_lb = lb[lb["religious_belief_at_nde"] == "christian"]
jesus_only = christian_lb[christian_lb["being_identifications"].apply(lambda x: "jesus" in x and "god" not in x and "unknown_presence" not in x)]
unknown_only = christian_lb[christian_lb["unknown_only"]]
god_only = christian_lb[christian_lb["being_identifications"].apply(lambda x: "god" in x and "jesus" not in x and "unknown_presence" not in x)]
print(f"Christian (belief at NDE) Light-Being encounters: {len(christian_lb)}")
print(f"  Jesus only {len(jesus_only)} | Unknown only {len(unknown_only)} | God only {len(god_only)} | other/mixed {len(christian_lb)-len(jesus_only)-len(unknown_only)-len(god_only)}")
jt = property_table(jesus_only, unknown_only, "Jesus-only", "Unknown-only")
print(jt.to_string(index=False, float_format=lambda v: f"{v:.3g}"))
print(f"\nProperties with |difference| ≥ 10 pp: {int((jt['diff (pp)'].abs() >= 10).sum())} of {len(jt)};  Holm-significant: {list(jt.loc[jt.p_holm < 0.05, 'property'])}")

Christian (belief at NDE) Light-Being encounters: 460
  Jesus only 102 | Unknown only 175 | God only 83 | other/mixed 100


                        property     Jesus-only    Unknown-only  diff (pp)   95% CI (pp)        p   p_holm
               guidance received 86/102 (84.3%) 144/175 (82.3%)          2  -7.0 to 11.1    0.741        1
               teaching guidance 24/102 (23.5%)  39/175 (22.3%)        1.2  -9.0 to 11.5    0.882        1
        telepathic communication 53/102 (52.0%)  88/175 (50.3%)        1.7 -10.5 to 13.9    0.804        1
                     life review 17/102 (16.7%)  50/175 (28.6%)      -11.9 -21.8 to -2.1   0.0292    0.262
         loving judgment | rated    5/9 (55.6%)    9/16 (56.2%)       -0.7 -41.3 to 39.9        1        1
      love tone | tone specified   3/11 (27.3%)   10/32 (31.2%)         -4 -34.8 to 26.9        1        1
       pleasant valence | stated  63/99 (63.6%)  87/175 (49.7%)       13.9   1.9 to 26.0   0.0317    0.262
light_encounter = being_of_light 64/102 (62.7%)  31/175 (17.7%)         45  34.1 to 56.0 4.11e-14 4.52e-13
                unity experience 47/1

In [8]:
print("Robustness: same comparison using religious_BACKGROUND = christian")
cb = lb[lb["religious_background"] == "christian"]
jt2 = property_table(cb[cb["being_identifications"].apply(lambda x: "jesus" in x and "god" not in x and "unknown_presence" not in x)],
                     cb[cb["unknown_only"]], "Jesus-only", "Unknown-only")
print(jt2.to_string(index=False, float_format=lambda v: f"{v:.3g}"))

Robustness: same comparison using religious_BACKGROUND = christian


                        property    Jesus-only    Unknown-only  diff (pp)   95% CI (pp)        p   p_holm
               guidance received 83/98 (84.7%) 179/217 (82.5%)        2.2  -6.5 to 10.9    0.745        1
               teaching guidance 24/98 (24.5%)  59/217 (27.2%)       -2.7  -13.1 to 7.7    0.679        1
        telepathic communication 55/98 (56.1%) 118/217 (54.4%)        1.7 -10.1 to 13.6    0.808        1
                     life review 15/98 (15.3%)  68/217 (31.3%)        -16 -25.5 to -6.6  0.00243   0.0219
         loving judgment | rated   4/7 (57.1%)   17/28 (60.7%)       -3.6 -44.5 to 37.3        1        1
      love tone | tone specified   2/9 (22.2%)   14/46 (30.4%)       -8.2 -38.5 to 22.0        1        1
       pleasant valence | stated 62/94 (66.0%) 116/217 (53.5%)       12.5   0.8 to 24.2   0.0461    0.369
light_encounter = being_of_light 64/98 (65.3%)  40/217 (18.4%)       46.9  36.1 to 57.6 8.04e-16 8.84e-15
                unity experience 44/98 (44.9%)

**Finding (mixed — the previous "all differences below 10%" is withdrawn).**

- **Similar** (small differences, not significant): guidance, teaching, telepathic communication, sense of belonging, mission commissioning, and — on small samples — judgment character and emotional tone. The *functional* profile of the encounter is statistically indistinguishable between the two labels. This is consistent with "constant state, variable form".
- **Different** (≥ 10 pp): Jesus-only encounters are far more often coded as a visual *being of light* (+45 pp); Unknown-only encounters report **more unity experience** (≈ +20 pp, significant after Holm correction) and more life reviews; Jesus-only report pleasant valence more often (nominal).
- The confidence intervals for the "similar" properties are about ±10 pp, so the data rule out large differences but not moderate ones.

*Interpretation:* naming covaries with the *mode* of the encounter (personified visual figure vs. formless presence with unity), not only with vocabulary. Whether the mode is part of the "form" the framework expects to vary, or part of the "state" it expects to be constant, has to be specified in advance for this to be a test.

In [9]:
print("DENOMINATIONS (Christian belief at NDE, Light-Being encounters; denomination stated)")
den = christian_lb[christian_lb["religious_belief_at_nde_denomination"].notna() & (christian_lb["religious_belief_at_nde_denomination"] != "not_specified")]
rows = []
for d_, g in den.groupby("religious_belief_at_nde_denomination"):
    rows.append({"denomination": d_, "n": len(g), "any Jesus %": round(100*nd.has(g, "being_identifications", "jesus").mean(), 1),
                 "unknown only %": round(100*g.unknown_only.mean(), 1), "any God %": round(100*nd.has(g, "being_identifications", "god").mean(), 1)})
print(pd.DataFrame(rows).sort_values("n", ascending=False).to_string(index=False))
cat = den[den.religious_belief_at_nde_denomination == "catholic"]
eva = den[den.religious_belief_at_nde_denomination == "evangelical_baptist"]
t = [[int(nd.has(cat, "being_identifications", "jesus").sum()), len(cat) - int(nd.has(cat, "being_identifications", "jesus").sum())],
     [int(nd.has(eva, "being_identifications", "jesus").sum()), len(eva) - int(nd.has(eva, "being_identifications", "jesus").sum())]]
orr, p = fisher_exact(t)
print(f"\nCatholic vs Evangelical/Baptist, any Jesus: {t[0][0]}/{len(cat)} vs {t[1][0]}/{len(eva)}; Fisher OR = {orr:.2f}, p = {p:.2f}")
print("With n = 24 Evangelicals the test can only detect very large differences; 'similar rates' is not evidence of equivalence.")

DENOMINATIONS (Christian belief at NDE, Light-Being encounters; denomination stated)


         denomination   n  any Jesus %  unknown only %  any God %
             catholic 120         21.7            51.7       23.3
      other_christian  71         35.2            40.8       28.2
  mainline_protestant  41         26.8            53.7       24.4
  evangelical_baptist  24         20.8            45.8       37.5
           mormon_lds  13         46.2            30.8       15.4
     jehovahs_witness   2         50.0            50.0        0.0
seventh_day_adventist   1          0.0           100.0        0.0

Catholic vs Evangelical/Baptist, any Jesus: 26/120 vs 5/24; Fisher OR = 1.05, p = 1.00
With n = 24 Evangelicals the test can only detect very large differences; 'similar rates' is not evidence of equivalence.


---
## Part VI: Predictive modelling (corrected)

The previous model predicted "God or Jesus" vs *everything else* among all being encounters (including 1,898 encounters with no Light Being at all), used `single_being` (a direct function of the identification list) as a feature, and then stated that `is_christian` was the top predictor although `single_being` had 85% of the importance. Here the question is asked cleanly: among Light-Being encounters with a known belief, how well does belief alone predict whether a named figure (rather than an unknown presence) is reported?

In [10]:
from sklearn.model_selection import StratifiedKFold, cross_validate
from sklearn.linear_model import LogisticRegression
from sklearn.dummy import DummyClassifier
k = lb[lb["religious_belief_at_nde"] != "not_mentioned"]
X = pd.get_dummies(k["religious_belief_at_nde"]).astype(float)
y = (~k["unknown_only"]).astype(int)
cv = StratifiedKFold(5, shuffle=True, random_state=0)
for name, model in [("prior (no information)", DummyClassifier(strategy="prior")), ("logistic regression on belief", LogisticRegression(max_iter=1000))]:
    r = cross_validate(model, X, y, cv=cv, scoring=["accuracy", "roc_auc", "neg_log_loss"])
    print(f"{name:32s} n={len(k)}  accuracy={r['test_accuracy'].mean():.3f}  AUC={r['test_roc_auc'].mean():.3f}  log-loss={-r['test_neg_log_loss'].mean():.4f}")

prior (no information)           n=719  accuracy=0.534  AUC=0.500  log-loss=0.6908


logistic regression on belief    n=719  accuracy=0.615  AUC=0.602  log-loss=0.6707


**Finding:** belief at the time of the NDE carries modest information about whether a named figure is reported (AUC clearly above 0.5 but far from deterministic). Culture shapes naming, partially.

---
## Part VII: Normative path markers

The normative-path markers previously duplicated here relied on v1 fields and hard-coded values. The corrected analysis is in **notebook 02**. Key corrected values for reference:

In [11]:
known_agency = df[df.return_agency != "not_mentioned"]
print(f"Deceased relatives encountered:        {nd.fmt_pct(int(nd.isin(df, 'deceased_relatives', nd.DECEASED_YES).sum()), N)}")
print(f"Return not self-chosen (stated):       {nd.fmt_pct(int(known_agency.return_agency.isin(['external_being','involuntary']).sum()), len(known_agency))}")
print(f"'Not your time' given as reason:      {nd.fmt_pct(int(nd.has(df, 'return_reasons', 'not_your_time').sum()), N)}")
print(f"Earthly mission or family as reason:   {nd.fmt_pct(int(nd.isin(df, 'return_reasons', ['earthly_mission','family_responsibility']).sum()), N)}")
print(f"Any communication with beings:         {nd.fmt_pct(int(df.communication_modes.apply(lambda ms: bool(set(ms) - {'no_communication'})).sum()), N)}")
print(f"Guidance received:                     {nd.fmt_pct(int((df.guidance_received=='yes').sum()), N)}")
print(f"Major or subtle value shift:           {nd.fmt_pct(int(df.value_shift.isin(['major','subtle']).sum()), N)}")
print(f"Reluctant to return (of stated):       {nd.fmt_pct(int((df.return_willingness=='reluctant').sum()), int((df.return_willingness!='not_mentioned').sum()))}")

Deceased relatives encountered:        1206/6751 = 17.9% [95% CI 17.0–18.8]
Return not self-chosen (stated):       3352/4781 = 70.1% [95% CI 68.8–71.4]
'Not your time' given as reason:      1459/6751 = 21.6% [95% CI 20.6–22.6]
Earthly mission or family as reason:   1649/6751 = 24.4% [95% CI 23.4–25.5]
Any communication with beings:         4378/6751 = 64.8% [95% CI 63.7–66.0]
Guidance received:                     3469/6751 = 51.4% [95% CI 50.2–52.6]
Major or subtle value shift:           3230/6751 = 47.8% [95% CI 46.7–49.0]
Reluctant to return (of stated):       1759/3563 = 49.4% [95% CI 47.7–51.0]


---
## Conclusion (corrected)

| Component of the theory | Corrected evidence | Assessment |
|---|---|---|
| Cultural vocabulary shapes naming | Belief predicts naming (V ≈ 0.2 in the 4-group test; AUC modest) | Supported — weak-to-moderate effect |
| A majority transcends labels | ~51% "unknown presence only"; Christians ~38–43% | Supported descriptively |
| Same properties under different names | Functional properties similar; visual-being coding (+45 pp) and unity experience (−20 pp) differ | **Partially supported** |
| Singular Being | Not measurable with this schema | Underdetermined |
| Loving rather than condemning | Harsh evaluation rare; ~50–60% loving, ~20–28% uncomfortable among rated | Supported for "non-condemning"; "unconditional love" overstated |
| Experiencer feels love rather than shame | Love ≈ shame; mixed is modal | Not supported |
| Personal, conscious entity | Communication and guidance are almost universal in Light-Being encounters | Consistent; personhood is an interpretation |
| Belief correction (fear ↓, spirituality ↑) | Strong where reported; fear reduction not specific to Light-Being encounters; religiosity does not decline | Supported for NDEs generally |
| Hell absent | 2.8% hellish realms; 3.7% distressing/terrifying | Uncommon, **not absent** |

---
**Notebook**: `04_conceptual_framework_theory.ipynb` · **Schema**: 2026-01-06 · **Audited**: 2026-10-05